# Error Handling & File Handling in Python

# Error Handling

## Syntax
```
try:
    # code that might raise an error
except ExceptionType:
    # handle the error
else:
    # runs if no exception occurred
finally:
    # always runs
```

## Basic try / except

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Error: Cannot divide by zero")

## Catching Multiple Exceptions

In [ ]:
def parse_record(value):
    try:
        result = int(value) * 2
        items = [1, 2, 3]
        print(items[result])
    except ValueError:
        print(f"ValueError: '{value}' cannot be converted to int")
    except IndexError:
        print(f"IndexError: index {result} is out of range")

parse_record("abc")
parse_record("5")

## else and finally

In [ ]:
def connect_to_db(host):
    try:
        if host == "":
            raise ValueError("Host cannot be empty")
        print(f"Connecting to {host}...")
    except ValueError as e:
        print(f"Connection error: {e}")
    else:
        print("Connection successful!")
    finally:
        print("Connection attempt finished.\n")

connect_to_db("")
connect_to_db("snowflake.us-east-1.aws")

## Catching the Exception Object (as e)

In [ ]:
try:
    data = {"pipeline": "ETL", "status": "running"}
    print(data["source"])          # key does not exist
except KeyError as e:
    print(f"Missing key: {e}")
except Exception as e:
    print(f"Unexpected error: {type(e).__name__} - {e}")

## Raising Custom Exceptions

In [ ]:
class DataValidationError(Exception):
    pass

def validate_row(row):
    if not isinstance(row.get("id"), int):
        raise DataValidationError(f"'id' must be an integer, got: {row.get('id')}")
    if not row.get("name"):
        raise DataValidationError("'name' field is required")
    return True

rows = [
    {"id": 1, "name": "Alice"},
    {"id": "two", "name": "Bob"},
    {"id": 3, "name": ""},
]

for row in rows:
    try:
        validate_row(row)
        print(f"Row {row['id']} is valid")
    except DataValidationError as e:
        print(f"Invalid row {row}: {e}")

# File Handling

## Syntax
```
# open modes
# 'r'  - read (default)
# 'w'  - write (overwrites)
# 'a'  - append
# 'r+' - read and write

with open("filename.txt", "mode") as f:
    # work with f
```
Using `with` automatically closes the file — no need to call `f.close()`.

## Writing to a File

In [ ]:
pipeline_logs = [
    "2024-01-01 08:00 | pipeline started",
    "2024-01-01 08:05 | extracted 1000 rows from source",
    "2024-01-01 08:10 | transformed data",
    "2024-01-01 08:15 | loaded to warehouse",
    "2024-01-01 08:16 | pipeline finished",
]

with open("pipeline.log", "w") as f:
    for log in pipeline_logs:
        f.write(log + "\n")

print("Log file written.")

## Reading a File

In [ ]:
# read() - entire file as one string
with open("pipeline.log", "r") as f:
    content = f.read()

print(content)

In [ ]:
# readlines() - list of lines
with open("pipeline.log", "r") as f:
    lines = f.readlines()

for line in lines:
    print(line.strip())

## Appending to a File

In [ ]:
with open("pipeline.log", "a") as f:
    f.write("2024-01-01 09:00 | next run started\n")

print("Appended to log file.")

## Handling File Errors

In [ ]:
try:
    with open("missing_file.csv", "r") as f:
        data = f.read()
except FileNotFoundError as e:
    print(f"File not found: {e}")
except PermissionError as e:
    print(f"Permission denied: {e}")

## Working with CSV Files

In [ ]:
import csv

employees = [
    {"name": "Alice", "role": "Data Engineer", "tool": "Spark"},
    {"name": "Bob",   "role": "Data Analyst",  "tool": "SQL"},
    {"name": "Carol", "role": "ML Engineer",   "tool": "Python"},
]

# write CSV
with open("employees.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "role", "tool"])
    writer.writeheader()
    writer.writerows(employees)

print("CSV written.")

# read CSV
with open("employees.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)

## Working with JSON Files

In [ ]:
import json

pipeline_config = {
    "pipeline_name": "daily_etl",
    "source": "S3",
    "destination": "Snowflake",
    "schedule": "0 6 * * *",
    "retries": 3,
    "tools": ["Airflow", "Spark", "dbt"]
}

# write JSON
with open("config.json", "w") as f:
    json.dump(pipeline_config, f, indent=4)

print("JSON written.")

# read JSON
with open("config.json", "r") as f:
    config = json.load(f)

print(f"Pipeline : {config['pipeline_name']}")
print(f"Source   : {config['source']}")
print(f"Tools    : {', '.join(config['tools'])}")

## Putting It All Together — ETL with Error Handling + File I/O

In [ ]:
import csv
import json

# sample source CSV with one bad row
source_data = "id,amount,currency\n1,500,USD\n2,abc,EUR\n3,300,GBP\n"

with open("transactions.csv", "w") as f:
    f.write(source_data)

errors = []
processed = []

try:
    with open("transactions.csv", "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                row["amount"] = int(row["amount"])
                processed.append(row)
            except ValueError:
                errors.append({"row": row, "error": f"Invalid amount: {row['amount']}"})

    print(f"Processed: {len(processed)} rows")
    print(f"Errors:    {len(errors)} rows\n")

    with open("processed.json", "w") as f:
        json.dump(processed, f, indent=4)

    with open("errors.log", "w") as f:
        for err in errors:
            f.write(str(err) + "\n")

    print("Results saved to processed.json and errors.log")

except FileNotFoundError as e:
    print(f"Source file not found: {e}")